In [ ]:
# ruff: noqa: T201, T203, ERA001, BLE001, E402
from __future__ import annotations

import logging
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

while Path.cwd().name in ["notebooks", "evaluation"]:
    os.chdir("..")

from config.runs_vars import SUBSET_TO_DATASET_MAP
from src.evaluate import reevaluate_setup
from src.utils import expand_setups

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

### Setups


In [ ]:
BASE_SETUP = {
    "subset": "human-mouse",
    "model": "gemini-2.0-flash-lite",
    "suffix": "_reduced",
    "experiment_type": "n_few_shots",
    "prompt_functions": [
        "direct_entity",
        "direct_entity_ontological",
        "direct_entity_with_synonyms",
        "sequential_hierarchy",
        "sequential_hierarchy_ontological",
        "sequential_hierarchy_with_synonyms",
    ],
}

SETUPS = [
    {
        "subset": list(SUBSET_TO_DATASET_MAP.keys()),
        "experiment_name": [
            f"v{v}/{n}" for v in range(10) for n in (0, 1, 2, 3, 5, 8, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75)
        ],
    }
]
SETUPS = expand_setups(SETUPS, BASE_SETUP, exclude_keys=["prompt_functions"])

print(f"Total setups to evaluate: {len(SETUPS)}")

In [ ]:
MAX_WORKERS = 5
SAVE_DIAGRAMS = False

futures = []
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for setup in SETUPS:
        futures.append(executor.submit(reevaluate_setup, setup, SAVE_DIAGRAMS))

    for f in tqdm(as_completed(futures), total=len(futures), desc="Processing setups"):
        result = f.result()
        if not result["success"]:
            print(f"Errors in {result['subset']} / {result['exp_name']}:")
            for err in result["errors"]:
                print("   ", err)
